# Multi-Benchmark Evaluation & Structural Ablation Study
## Upgraded 16-D Structural Spatial Backbone on FUNSD, XFUND, and VRDU

**IE 643 Course Project** · Rigorous Evaluation on Dual Tesla T4 GPUs

---

### 1. Architecture Overview: Upgraded 16-D Spatial Inductive Bias
To solve the "jagged table" and multi-line box problems without sacrificing zero-shot cross-template generalization, the spatial relation extraction head incorporates 4 distinct feature groups:

| Feature Group | Dims | Mathematical Formulation | Hypothesis & Inductive Bias |
|---|---|---|---|
| **1. Continuous Relative Geometry** | 5 | $\Delta x / w_A, \Delta y / h_A, d_{\text{euc}} / D, \cos\theta, \sin\theta$ | Directional bearing and normalized Euclidean distance. |
| **2. Axial Containment Overlap-over-Min** | 5 | $\frac{\text{inter}_x}{\min(w_A, w_B)}, \frac{\text{inter}_y}{\min(h_A, h_B)}, \text{is\_disjoint}_x, \text{is\_disjoint}_y, \text{is\_downward}$ | Solves multi-line values (e.g. 4-line address) where standard Jaccard IoU fails due to tall bounding boxes. |
| **3. Line-of-Sight (LOSG) Visibility** | 2 | $\text{LOS}_x, \text{LOS}_y \in \{0, 1\}$ (ray-casting corridor) | Solves jagged tables: links across empty space while blocking links obstructed by intervening text. |
| **4. Typographical Scale Hierarchy** | 4 | $\ln(h_B / h_A), \ln(A_B / A_A), z_{\text{font}}(A), z_{\text{font}}(B)$ | Identifies document hierarchy (headers vs keys vs values) via document-standardized font heights. |

---

### 2. Systematic Ablation Study Plan
We conduct a controlled leave-one-out and feature isolation ablation suite:
1. **Full 16-D Model** (All 4 groups active)
2. **Ablation 1 (w/o Line-of-Sight, 14-D)**: Disables LOS ray-casting to test impact on distant/jagged alignment.
3. **Ablation 2 (w/o Typography, 12-D)**: Disables font scale hierarchy to test impact of typographical prominence.
4. **Ablation 3 (w/o Containment, 11-D)**: Disables Overlap-over-Minimum to test multi-line box handling.
5. **Ablation 4 (Geometry Only, 5-D)**: Pure Euclidean/polar baseline without structural inductive biases.
6. **Ablation 5 (Loss Function: BCE vs Focal)**: Evaluates canonical focal loss vs binary cross-entropy on $O(N^2)$ candidate sparsity.
7. **Ablation 6 (Decoding: Independent vs 1-to-1 Argmax)**: Evaluates the single-key assignment constraint.

## 0 · Environment Setup & Git Clone

In [ ]:
# Clone or pull latest KeyValue-Extraction repository
import os, subprocess, sys

REPO_URL = "https://github.com/ABasakEE/KeyValue-Extraction.git"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest main branch...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Ensure project root and src/ are in sys.path
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Python sys.path initialized: {sys.path[:2]}")

In [ ]:
# Install required packages
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval faker transformers datasets

In [ ]:
# Verify GPU hardware
import torch
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()} | GPU Count: {gpu_count}")
if gpu_count > 0:
    for i in range(gpu_count):
        prop = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}] {prop.name} | VRAM: {prop.total_memory / 1e9:.2f} GB")
    if gpu_count >= 2:
        print("\n[+] DUAL GPU ACTIVE (Tesla T4 x 2) -> Multi-GPU seed parallelism enabled!")
    else:
        print("\n[*] Single GPU active -> Training sequentially.")
else:
    print("WARNING: Running on CPU. GPU recommended.")

## 1 · Dataset Downloads & Preprocessing
Downloads and configures XFUND (5 Latin languages), FUNSD, and VRDU.

In [ ]:
import os, urllib.request

# 1. Download XFUND (5 Latin languages: es, fr, de, it, pt)
xfund_dir = "data/xfund"
os.makedirs(xfund_dir, exist_ok=True)
BASE_URL = "https://github.com/doc-analysis/XFUND/releases/download/v1.0/"
LANGUAGES = ["es", "fr", "de", "it", "pt"]

print("Checking XFUND dataset...")
for lang in LANGUAGES:
    for split in ["train", "val"]:
        filename = f"{lang}.{split}.json"
        target_path = os.path.join(xfund_dir, filename)
        if not os.path.isfile(target_path) or os.path.getsize(target_path) < 1000:
            url = BASE_URL + filename
            print(f"  Downloading {filename}...")
            urllib.request.urlretrieve(url, target_path)
        print(f"  [OK] {filename} ({os.path.getsize(target_path) / 1024:.1f} KB)")

print("\nXFUND verification complete!")

In [ ]:
# 2. Verify FUNSD dataset
funsd_dir = "data/funsd/dataset"
if os.path.isdir(funsd_dir):
    print(f"[OK] FUNSD dataset found at {funsd_dir}")
else:
    print(f"Downloading FUNSD dataset from HuggingFace / public mirror...")
    os.makedirs(funsd_dir, exist_ok=True)
    # Extract or download FUNSD if needed
    subprocess.run(["python", "-c", """
from datasets import load_dataset
import json, os
try:
    ds = load_dataset('nielsr/funsd')
    print('Successfully verified FUNSD dataset via HuggingFace')
except Exception as e:
    print(f'HuggingFace FUNSD check: {e}')
"""], check=False)

# 3. Verify VRDU registration-form dataset
vrdu_dir = "data/vrdu/registration-form"
if os.path.isdir(vrdu_dir):
    print(f"[OK] VRDU dataset found at {vrdu_dir}")
else:
    print(f"[*] VRDU directory at {vrdu_dir} (will download if running Track B)")

## 2 · Architectural Unit Tests: 16-D Features, LOSG & Dynamic Ablation

In [ ]:
# Run all unit tests for the 16-D spatial features, LOS visibility, and ablation logic
!python -m unittest tests/test_linking_head.py

---
## 3 · Primary Baseline: Full 16-D Upgraded Model on Scaled XFUND

Runs the complete **16-D Upgraded Structural Backbone**:
- Continuous Geometry (5D) + Axial Containment Overlap-over-Min (5D) + LOS Visibility (2D) + Typographical Hierarchy (4D)
- Normalized Focal Loss ($N_{\text{pos}}$)
- One-to-One Argmax Decoding
- Decoupled 3-Stage Curriculum with Early Stopping (Patience = 3)
- Multi-GPU Seed Parallelism (Dual Tesla T4)

In [ ]:
!python src/train.py --config configs/xfund_joint_lilt.yaml --output-dir results/xfund_joint_16d

## 4 · Baseline on FUNSD Benchmark (16-D Upgraded Model)

In [ ]:
!python src/train.py --config configs/funsd_joint_lilt_curriculum.yaml --output-dir results/funsd_joint_16d

---
## 5 · Systematic Feature Group Ablation Suite

To isolate the exact contribution of each component to cross-template generalization and resolving jagged tables, we execute the following controlled ablations on XFUND.

In [ ]:
ABLATIONS = [
    {
        "name": "Ablation 1: w/o Line-of-Sight Visibility (14-D)",
        "desc": "Disables corridor ray-casting to test impact on distant / jagged column headers.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --no-line-of-sight --output-dir results/ablation_no_los"
    },
    {
        "name": "Ablation 2: w/o Typographical Scale Hierarchy (12-D)",
        "desc": "Disables font size / area log ratio and document z-score prominence.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --no-typography --output-dir results/ablation_no_typography"
    },
    {
        "name": "Ablation 3: w/o Axial Containment Overlap-over-Min (11-D)",
        "desc": "Disables min-based overlap to test multi-line text box failure.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --no-containment --output-dir results/ablation_no_containment"
    },
    {
        "name": "Ablation 4: Continuous Geometry Only (5-D)",
        "desc": "Pure continuous polar / distance baseline without structural inductive biases.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --geometry-only --output-dir results/ablation_geometry_only"
    },
    {
        "name": "Ablation 5: Loss Function Ablation (BCE vs Focal Loss)",
        "desc": "Evaluates standard BCE loss without focal hard negative modulation.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --loss-type bce --output-dir results/ablation_bce_loss"
    },
    {
        "name": "Ablation 6: Decoding Strategy (Independent vs One-to-One Argmax)",
        "desc": "Disables 1-to-1 argmax matching to test candidate link collisions.",
        "cmd": "python src/train.py --config configs/xfund_joint_lilt.yaml --no-one-to-one --output-dir results/ablation_no_one_to_one"
    },
]

print("Available Ablation Experiments:")
for i, ab in enumerate(ABLATIONS):
    print(f"[{i+1}] {ab['name']}")
    print(f"    {ab['desc']}")

In [ ]:
# Execute the ablations sequentially (with multi-GPU seed parallelism inside each run)
import subprocess, sys

for i, ab in enumerate(ABLATIONS):
    print(f"\n{'='*75}")
    print(f"[*] RUNNING {ab['name']}")
    print(f"[*] Command: {ab['cmd']}")
    print(f"{'='*75}\n", flush=True)
    ret = subprocess.run(ab['cmd'], shell=True)
    if ret.returncode != 0:
        print(f"[-] Error running {ab['name']}, return code {ret.returncode}")
    else:
        print(f"[+] Successfully completed {ab['name']}")

---
## 6 · Comprehensive Results & Comparison Table

Aggregates `metrics.json` across all conditions, computing mean, standard deviation, and delta against the previous 12-D baseline (65.51% F1).

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Previous baseline calibration from 12-D run
PREV_BASELINE_F1 = 0.6551
PREV_ENTITY_F1 = 0.8407

runs_to_eval = [
    ("Previous 12-D Baseline (Reference)", "results/xfund_joint_lilt"),
    ("Full 16-D Upgraded Model", "results/xfund_joint_16d"),
    ("Ablation: w/o Line-of-Sight (14-D)", "results/ablation_no_los"),
    ("Ablation: w/o Typography (12-D)", "results/ablation_no_typography"),
    ("Ablation: w/o Containment Overlap-over-Min (11-D)", "results/ablation_no_containment"),
    ("Ablation: Geometry Only (5-D)", "results/ablation_geometry_only"),
    ("Ablation: BCE Loss (no Focal)", "results/ablation_bce_loss"),
    ("Ablation: Independent Decoding (no 1-to-1)", "results/ablation_no_one_to_one"),
    ("FUNSD Joint 16-D Upgraded", "results/funsd_joint_16d"),
]

records = []
for label, path_str in runs_to_eval:
    mf = Path(path_str) / "metrics.json"
    if mf.exists():
        data = json.loads(mf.read_text(encoding="utf-8"))
        ent_f1 = data.get("mean_f1", 0.0)
        ent_std = data.get("std_f1", 0.0)
        lnk_f1 = data.get("mean_linking_f1", 0.0)
        lnk_std = data.get("std_linking_f1", 0.0)
        delta = lnk_f1 - PREV_BASELINE_F1 if "FUNSD" not in label else 0.0
        records.append({
            "Architecture / Condition": label,
            "Entity F1": f"{ent_f1*100:.2f}% ± {ent_std*100:.2f}%",
            "Linking F1": f"{lnk_f1*100:.2f}% ± {lnk_std*100:.2f}%",
            "Δ vs 12-D Baseline": f"{delta*100:+.2f}%" if "FUNSD" not in label else "N/A",
            "Status": "Completed"
        })
    else:
        records.append({
            "Architecture / Condition": label,
            "Entity F1": "--",
            "Linking F1": "--",
            "Δ vs 12-D Baseline": "--",
            "Status": "Pending / Not Run"
        })

df = pd.DataFrame(records)
print("="*85)
print("           CROSS-BENCHMARK ABLATION & EVALUATION SUMMARY")
print("="*85)
display(df)

# Render Markdown table for copy-paste into report
print("\n--- Markdown Table ---")
print(df.to_markdown(index=False))

---
## 7 · Package Outputs for Local Download

Compresses all results, metrics, and models into `kaggle_ablation_results.zip` in `/kaggle/working`.

In [ ]:
!cd /kaggle/working && zip -q -r /kaggle/working/kaggle_ablation_results.zip KeyValue-Extraction/results -x "*.pyc" "*__pycache__*"
print("[+] Packaging complete: /kaggle/working/kaggle_ablation_results.zip")
!ls -lh /kaggle/working/kaggle_ablation_results.zip